In [4]:
import pandas as pd
import sqlite3

# Load the two Kaggle files
df_freq = pd.read_csv('../data/freMTPL2freq.csv')
df_sev = pd.read_csv('../data/freMTPL2sev.csv')

# Merge them on the policy ID column (commonly 'IDpol')
df = df_freq.merge(df_sev[['IDpol', 'ClaimAmount']], on='IDpol', how='left')

# Fill missing ClaimAmount with 0 for policies that had no claims
df['ClaimAmount'] = df['ClaimAmount'].fillna(0)

# Sanity check
print(f"Total rows: {df.shape[0]}")
print(df[['IDpol', 'Exposure', 'ClaimNb', 'ClaimAmount']].head())

# Optional: save the merged file for future use
df.to_csv('../data/freMTPL2.csv', index=False)

# Create SQLite database and load the table
conn = sqlite3.connect('../data/motor_insurance.db')
df.to_sql('policies', conn, if_exists='replace', index=False)
conn.close()

print("✅ Database 'motor_insurance.db' created with 'policies' table.")


Total rows: 679513
   IDpol  Exposure  ClaimNb  ClaimAmount
0    1.0      0.10        1          0.0
1    3.0      0.77        1          0.0
2    5.0      0.75        1          0.0
3   10.0      0.09        1          0.0
4   11.0      0.84        1          0.0
✅ Database 'motor_insurance.db' created with 'policies' table.


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/motor_insurance.db')

query1 = '''
SELECT
    COUNT(*) AS total_policies,
    SUM(Exposure) AS total_exposure,
    SUM(ClaimNb) AS total_claims,
    ROUND(CAST(SUM(ClaimNb) AS REAL) / SUM(Exposure), 3) AS overall_claim_frequency
FROM policies;
'''
df1 = pd.read_sql(query1, conn)
print(df1)

conn.close()

   total_policies  total_exposure  total_claims  overall_claim_frequency
0          679513   359515.229161         39788                    0.111
